# AI-Generated Image Detection Using Machine Learning and Deep Learning

Comparison of three detectors of AI-generated images: **SVM** and **Random Forest** on 2048-D ResNet50
(ImageNet1K_V1) features, and an end-to-end fine-tuned **ResNet50 CNN**.

**Experiments**
- **Exp 1:** CIFAKE train → CIFAKE test
- **Exp 2:** GenImage (ADM, GLIDE, SD1.5, VQDM, Wukong) → held-out BigGAN and Midjourney
- **Exp 3:** Exp 1 models evaluated on the Exp 2 test set (no retraining)
- **Robustness:** JPEG (Q90/70/50), resizing (112, 56) and Gaussian noise (σ = 0.01/0.03/0.05) on the Exp 1 and Exp 2 test sets
- **32×32:** Exp 2 protocol with all images at 32×32 resolution

All shared settings are in `config.py`; the source code is in the project folders (see `README.md`).
Every step skips work whose outputs already exist, so the pipeline can be resumed.

## 1. Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/AI_Image_Detection'
os.environ['AIID_CACHE_DIR'] = '/content/cache'            # preprocessed images (local disk)
os.environ['AIID_CIFAKE_ROOT'] = '/content/data/CIFAKE'
os.environ['AIID_GENIMAGE_ROOT'] = '/content/data/GenImage'
%cd {PROJECT}

In [ ]:
!pip install -q imagehash kagglehub datasets
!python utils.py
!python -m evaluation.metrics
!python -m evaluation.transforms

## 2. Data acquisition
- **CIFAKE:** Kaggle `birdy654/cifake-real-and-ai-generated-synthetic-images`
- **GenImage:** Hugging Face subset `TheKernel01/Tiny-GenImage` (SD1.4 is not included in this subset)

In [ ]:
import kagglehub
os.environ['AIID_CIFAKE_ROOT'] = kagglehub.dataset_download('birdy654/cifake-real-and-ai-generated-synthetic-images')
if not os.path.exists('/content/data/GenImage/Midjourney'):
    !python -m preprocessing.export_tiny_genimage
!python -m preprocessing.genimage_dataset --check

## 3. Sampling, cleaning and preprocessing
Balanced sampling (CIFAKE 20,000; GenImage 17,500), corrupted-file detection, SHA-256 and pHash duplicate
removal, JPEG Q95 re-encoding of GenImage, resizing to 224×224. The preprocessed cache is backed up to Drive.

In [ ]:
import shutil, glob
CACHE, BACKUP = '/content/cache', f'{PROJECT}/data/cache_backup'
os.makedirs(CACHE, exist_ok=True); os.makedirs(BACKUP, exist_ok=True)
for f in glob.glob(f'{BACKUP}/*.npy'):
    if not os.path.exists(os.path.join(CACHE, os.path.basename(f))):
        shutil.copy(f, CACHE)
!python -m preprocessing.cifake_dataset
!python -m preprocessing.genimage_dataset
for f in glob.glob(f'{CACHE}/*.npy'):
    if not os.path.exists(os.path.join(BACKUP, os.path.basename(f))):
        shutil.copy(f, BACKUP)

## 4. Leakage audit and exploratory data analysis

In [ ]:
!python -m preprocessing.leakage_audit
!python -m eda.run_eda

## 5. ResNet50 feature extraction (GPU)

In [ ]:
!python -m features.extract_features

## 6. CNN training and evaluation (GPU)

In [ ]:
!python -m training.train_cnn --tag cifake genimage res32
!python -m training.eval_cnn

## 7. SVM and Random Forest (CPU)

In [ ]:
!python -m models.svm_model
!python -m models.rf_model

## 8. Evaluation, robustness analysis and report

In [ ]:
!python -m evaluation.evaluate_models
!python -m evaluation.evaluate_robustness
!python -m evaluation.report_tables

## 9. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display
display(pd.read_csv('results/tables/final_comparison.csv').round(4))
for f in ['metric_comparison', 'confusion_matrices', 'roc_curves', 'robustness_comparison', 'training_curves']:
    display(Image(f'results/figures/{f}.png'))

## 10. Completeness check

In [ ]:
ok = True
def check(name, cond):
    global ok
    print(('PASS  ' if cond else 'FAIL  ') + name)
    ok = ok and cond

audit = pd.read_csv('data/manifests/leakage_audit.csv')
check('leakage audit has no failed checks', (audit['result'] == 'FAIL').sum() == 0)
for m in ['svm', 'rf', 'cnn']:
    p = f'results/predictions/{m}_predictions.csv'
    check(f'{m} predictions complete', os.path.exists(p) and len(pd.read_csv(p)) == 91000)
comp = pd.read_csv('results/tables/result_completeness.csv')
check('all experiments complete', (comp['status'] == 'MISSING').sum() == 0)
for m in ['svm', 'rf']:
    for t in ['cifake', 'genimage', 'res32']:
        check(f'checkpoint {m}_{t}', os.path.exists(f'checkpoints/{m}/{m}_{t}.joblib'))
for t in ['cifake', 'genimage', 'res32']:
    check(f'checkpoint cnn_{t}', os.path.exists(f'checkpoints/cnn/cnn_{t}.pt'))
print('\nAll checks passed.' if ok else '\nSome checks failed.')